# Stage 2 — Instruction Fine-Tuning (SFT)

This notebook continues from the Stage 1 LoRA adapter and trains on 132 policy-grounded instruction-response pairs. The dataset is formatted as conversational prompt-completion records, so loss is focused on the assistant completion.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/domain-ai-assistant-finetuning"

!rm -rf /content/domain-ai-assistant-finetuning
!ln -s "{PROJECT}" /content/domain-ai-assistant-finetuning

%cd /content/domain-ai-assistant-finetuning

Mounted at /content/drive
/content/drive/MyDrive/domain-ai-assistant-finetuning


In [2]:
# Run on a Google Colab GPU runtime (T4 or better).
!pip -q install -U unsloth trl transformers datasets peft accelerate bitsandbytes sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 140.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [3]:
from pathlib import Path
import json, os, sys, torch

# Expected workflow: clone the GitHub repository, then open the notebook from the repo.
CANDIDATES = [Path.cwd(), Path('/content/domain-ai-assistant-finetuning')]
ROOT = next((p for p in CANDIDATES if (p / 'data').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Repository root not found. Clone the repo into /content/domain-ai-assistant-finetuning or run from the repo root.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('Repository:', ROOT)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available')
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU runtime is required for these notebooks.')

Repository: /content/drive/MyDrive/domain-ai-assistant-finetuning
CUDA: True Tesla T4


In [4]:
# Unsloth must be imported before TRL, PEFT and Transformers.
from unsloth import FastLanguageModel, is_bfloat16_supported

from datasets import Dataset
from peft import PeftModel
from trl import SFTConfig, SFTTrainer

from common import (
    SYSTEM_PROMPT,
    generate_answer,
    read_jsonl,
    write_comparison_report,
)

BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024
STAGE1_ADAPTER = ROOT / "outputs/non_instruction_adapter"
SFT_ADAPTER = ROOT / "outputs/sft_adapter"

if not STAGE1_ADAPTER.exists():
    raise FileNotFoundError(
        "Stage 1 adapter was not found. Run non_instruction_finetuning.ipynb first."
    )

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!


## 1. Load and validate the instruction dataset

In [5]:
rows = read_jsonl(ROOT / 'data/instruction_dataset.jsonl')
assert len(rows) >= 100
assert len({r['instruction'] for r in rows}) == len(rows)
print('Instruction examples:', len(rows))
print(rows[0])

Instruction examples: 132
{'instruction': 'I forgot my corporate password. What should I do?', 'response': 'Use the NovaDesk self-service identity portal first. Verify with your employee ID and an approved factor, then create a new password. If verification fails, contact the Service Desk from a known phone number so your identity can be checked. A temporary password, when issued, expires after 30 minutes and must be changed at first sign-in. Never send an existing password in email, chat, or a ticket.', 'topic': 'Password reset'}


In [6]:
formatted = []
for row in rows:
    formatted.append({
        'prompt': [
            {'role':'system', 'content':SYSTEM_PROMPT},
            {'role':'user', 'content':row['instruction']},
        ],
        'completion': [{'role':'assistant', 'content':row['response']}],
    })
dataset = Dataset.from_list(formatted).train_test_split(test_size=0.1, seed=42)
print(dataset)
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 118
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 14
    })
})
{'prompt': [{'role': 'system', 'content': 'You are NovaDesk, an internal IT helpdesk assistant. Give safe, concise, policy-grounded troubleshooting steps. Never request or expose passwords, one-time codes, private keys, full payment-card numbers, health records, or unredacted identity documents. State when the user should stop troubleshooting and call the Service Desk or Security Operations. Do not invent outage restoration times.'}, {'role': 'user', 'content': 'My external monitor says no signal. What should I check?'}], 'completion': [{'role': 'assistant', 'content': 'Check monitor power and input source, reseat the cable and dock, verify the display mode, and test one known-good cable or port. Reboot the dock and laptop. Use only approved firmware and drivers. Include monitor model, 

## 3. Reload the base model and attach the Stage 1 adapter as trainable

In [7]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Correct Qwen2.5 tokens
tokenizer.eos_token = "<|im_end|>"
tokenizer.pad_token = "<|endoftext|>"

model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

if hasattr(model, "generation_config"):
    model.generation_config.eos_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id

# Continue training the Stage 1 LoRA adapter
model = PeftModel.from_pretrained(
    model,
    str(STAGE1_ADAPTER),
    is_trainable=True,
)

print("Loaded Stage 1 adapter:", STAGE1_ADAPTER)
print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)

model.print_trainable_parameters()

==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded Stage 1 adapter: /content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/non_instruction_adapter
EOS token: <|im_end|>
PAD token: <|endoftext|>
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 4. Train the SFT adapter

In [8]:
sft_args = SFTConfig(
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,

    # Required for Qwen2.5
    eos_token="<|im_end|>",

    output_dir=str(ROOT / "outputs/sft_training"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=4,
    learning_rate=1e-4,
    warmup_steps=2,
    logging_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    seed=42,
    report_to="none",
)

print("Tokenizer EOS:", tokenizer.eos_token)
print("SFTConfig EOS:", sft_args.eos_token)

assert tokenizer.eos_token == "<|im_end|>"
assert sft_args.eos_token == "<|im_end|>"

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=sft_args,
)

train_result = trainer.train()
print(train_result)

(ROOT / "artifacts/sft_train_metrics.json").write_text(
    json.dumps(train_result.metrics, indent=2, default=str),
    encoding="utf-8",
)

trainer.save_state()

Tokenizer EOS: <|im_end|>
SFTConfig EOS: <|im_end|>


Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=6):   0%|          | 0/118 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=6):   0%|          | 0/14 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 118 | Num Epochs = 4 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,2.665145,2.440306
2,1.763565,1.947218
3,1.324610,1.708927
4,1.148237,1.668057


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/sft_training/checkpoint-15/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/sft_training/checkpoint-30/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/sft_training/checkpoint-45/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/sft_training/checkpoint-60/tokenizer_config.json.


TrainOutput(global_step=60, training_loss=1.8923262238502503, metrics={'train_runtime': 168.9279, 'train_samples_per_second': 2.794, 'train_steps_per_second': 0.355, 'total_flos': 585142104932352.0, 'train_loss': 1.8923262238502503, 'epoch': 4.0})


## 5. Save, evaluate, and generate the SFT comparison report

In [9]:
SFT_ADAPTER.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SFT_ADAPTER)
tokenizer.save_pretrained(SFT_ADAPTER)
FastLanguageModel.for_inference(model)
questions = json.loads((ROOT / 'data/evaluation_questions.json').read_text())
sft_answers = [generate_answer(model, tokenizer, q) for q in questions]
(ROOT / 'artifacts/sft_outputs.json').write_text(json.dumps(sft_answers, indent=2), encoding='utf-8')
base_path = ROOT / 'artifacts/base_outputs.json'
base_answers = json.loads(base_path.read_text()) if base_path.exists() else ['Run Stage 1'] * len(questions)
write_comparison_report(
    ROOT / 'reports/sft_model_comparison.md',
    'Base Model vs Instruction Fine-Tuned Model',
    questions,
    [
        ('Base model answer', base_answers),
        ('Fine-tuned model answer', sft_answers),
        ('Which is better?', ['Complete after human review'] * len(questions)),
        ('Reason', ['Compare correctness, domain accuracy, clarity, safety, and helpfulness'] * len(questions)),
    ],
)
print('Saved adapter:', SFT_ADAPTER)
print('Updated reports/sft_model_comparison.md')

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/sft_adapter/tokenizer_config.json.
Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Trans

Saved adapter: /content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/sft_adapter
Updated reports/sft_model_comparison.md


In [10]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/domain-ai-assistant-finetuning")

required = [
    ROOT / "outputs/sft_adapter/adapter_config.json",
    ROOT / "outputs/sft_adapter/adapter_model.safetensors",
    ROOT / "outputs/sft_adapter/tokenizer_config.json",
    ROOT / "artifacts/sft_outputs.json",
    ROOT / "artifacts/sft_train_metrics.json",
    ROOT / "reports/sft_model_comparison.md",
]

for path in required:
    print("✅" if path.exists() else "❌", path)

✅ /content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/sft_adapter/adapter_config.json
✅ /content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/sft_adapter/adapter_model.safetensors
✅ /content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/sft_adapter/tokenizer_config.json
✅ /content/drive/MyDrive/domain-ai-assistant-finetuning/artifacts/sft_outputs.json
✅ /content/drive/MyDrive/domain-ai-assistant-finetuning/artifacts/sft_train_metrics.json
✅ /content/drive/MyDrive/domain-ai-assistant-finetuning/reports/sft_model_comparison.md


In [11]:
import json

questions = json.loads(
    (ROOT / "data/evaluation_questions.json").read_text()
)

answers = json.loads(
    (ROOT / "artifacts/sft_outputs.json").read_text()
)

for question, answer in zip(questions[:5], answers[:5]):
    print("=" * 80)
    print("QUESTION:")
    print(question)
    print("\nSFT ANSWER:")
    print(answer)

QUESTION:
My account is locked and keeps locking again after support unlocks it. What should I do?

SFT ANSWER:
Verify the correct account name and password. If you know the unlock code, restart the device and sign in with local administrator credentials. If the device is locked remotely, contact the Service Desk immediately so a manager can review the incident and investigate the lock reason. Do not install unknown remote unlocking tools.
QUESTION:
I clicked a suspicious email link and entered my corporate password. What actions should I take now?

SFT ANSWER:
Do not open attachments or click on unknown links in email or social media messages. Use the approved antivirus client and scan all removable media. If prompted, enter the business version of your Microsoft account and device name. If you cannot connect immediately, reboot and test one permitted approved network share.
QUESTION:
The VPN gives an error and will not connect from home. What should I check and include in the ticket?